# 2-2. 벡터스토어 비교: Chroma / FAISS / Qdrant

## 학습 목표
- 폐쇄망에서 쓸 수 있는 세 가지 벡터스토어의 특징과 제약사항을 구분할 수 있다.
- Chroma · FAISS · Qdrant의 **영속화 / 인메모리** 모드 사용법을 익힌다.
- 동일 코퍼스를 세 스토어에 적재해 **인덱싱 시간, 질의 시간, 결과 일치도**를 비교한다.
- 스토어별 **score 반환 의미의 차이**(거리 vs 유사도)를 구분해 읽을 수 있다.

## 핵심 키워드
`Chroma` `FAISS` `Qdrant` `persist_directory` `save_local` `in-memory` `payload filter`

## 폐쇄망 적합성 배지
| 스토어 | 설치 | 폐쇄망 | 영속화 | 메타 필터 | 배지 |
|---|---|---|---|---|---|
| **Chroma** | `pip` 로컬 DB (SQLite + Parquet) | 완전 OK | `persist_directory=...` | JSON 기반 `$and`/`$or` | 🔒 |
| **FAISS** | `faiss-cpu`, 벡터만 관리 | 완전 OK | `save_local / load_local` | 수동(post-filter) | 🔒 |
| **Qdrant** | 인메모리 또는 Docker/바이너리 | 완전 OK | `:memory:` 또는 `path=...` | 풍부한 `Filter` DSL | 🔒 |
| ~~Pinecone / Weaviate Cloud~~ | 매니지드 SaaS | ❌ 불가 | — | — | ☁️ |

> 🔑 **왜 이 세 개인가**: 모두 **인터넷 없이 동작**하는 로컬 스토어이고, LangChain이 통일된 `VectorStore` 인터페이스를 제공한다. 셋의 핵심 차이는 ① **운영 성격**(임베드형 vs 서버형), ② **메타데이터 필터링 능력**, ③ **영속화 파일 포맷**이다. 이 세 축으로 실무 선택이 갈린다.

> ⚠️ **Score는 서로 직접 비교 금지**: Chroma/FAISS는 **거리(L2 등)** 를 반환 → **작을수록 가깝다**. Qdrant는 이 실습에서 cosine **유사도** → **클수록 가깝다**. 같은 코퍼스라도 숫자의 의미가 반대이므로, 스토어 교체 시 랭킹 로직을 반드시 확인해야 한다.

In [ ]:
import sys; sys.path.insert(0, '../..')
from common import get_chat_model, get_embeddings, provider_badge
print(provider_badge())

In [ ]:
from pathlib import Path
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 로컬 코퍼스: data/corpus_ko.txt — 금융·RAG 관련 한국어 용어 정리 파일
corpus_text = Path('../../data/corpus_ko.txt').read_text(encoding='utf-8')
# 청크 수가 너무 적으면 스토어 간 속도 차이가 안 보이므로 인위적으로 5배 복제
corpus_text = (corpus_text + '\n') * 5

# chunk_size=300 / overlap=50 은 2-1 노트북에서 검증한 한국어 친화 설정
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.create_documents([corpus_text], metadatas=[{'source': 'corpus_ko.txt'}])

# 실습용 메타 부여 — 필터링 비교에 쓴다 (짝수=finance / 홀수=compliance)
for i, c in enumerate(chunks):
    c.metadata['chunk_id'] = i
    c.metadata['topic'] = 'finance' if i % 2 == 0 else 'compliance'

print(f'인덱싱 대상 청크: {len(chunks)}개')

# get_embeddings()는 lru_cache 되어 있으므로 세 스토어가 같은 임베딩 모델을 공유 → 비교 공정성 확보
emb = get_embeddings()

## 1. Chroma 🔒

**임베디드 DB** 타입. 별도 서버 프로세스 없이 파이썬 프로세스 안에서 동작한다.

- **저장 포맷**: `persist_directory` 하위에 `chroma.sqlite3`(메타/인덱스 스키마) + 컬렉션별 `{uuid}/*.bin`(근사 최근접 인덱스 + 벡터 바이너리).
- **운영 관점**: 단일 프로세스가 직접 접근하므로 **다중 쓰기(concurrent write)** 에 약함. 읽기는 여러 프로세스가 나눠도 됨.
- **폐쇄망 적합성**: 파일만 있으면 동작 → 인덱스 디렉터리 통째로 반입/반출 가능.

In [ ]:
import time, shutil
from langchain_community.vectorstores import Chroma

CHROMA_DIR = Path('./_store/chroma')
# 실습 재현성을 위해 기존 인덱스 폴더 제거 후 새로 빌드
if CHROMA_DIR.exists():
    shutil.rmtree(CHROMA_DIR)
CHROMA_DIR.parent.mkdir(parents=True, exist_ok=True)

t0 = time.time()
chroma_vs = Chroma.from_documents(
    documents=chunks,
    embedding=emb,
    persist_directory=str(CHROMA_DIR),   # 디스크에 영속 → 재시작 시 재인덱싱 불필요
    collection_name='efinance_terms',    # 한 디렉터리에 여러 컬렉션 공존 가능
)
chroma_index_sec = time.time() - t0
print(f'[Chroma] index 시간: {chroma_index_sec:.3f}s, 문서 수: {chroma_vs._collection.count()}')

In [ ]:
q = '청약철회 기간은 며칠인가?'
t0 = time.time()
# similarity_search_with_score — (Document, score) 튜플 반환. Chroma의 score는 기본적으로 L2 거리 (작을수록 유사)
res = chroma_vs.similarity_search_with_score(q, k=3)
chroma_q_sec = time.time() - t0
print(f'[Chroma] query: {chroma_q_sec*1000:.1f}ms')
for doc, score in res:
    print(f'  L2={score:.3f} topic={doc.metadata["topic"]}  {doc.page_content[:60]}…')

## 2. FAISS 🔒

Meta(구 Facebook)에서 만든 **벡터 전용 인덱스 라이브러리**. DB가 아니라 벡터 배열 위에서의 근사 최근접(ANN) 검색만 제공.

- **특징**: 메타데이터 저장소가 별도로 없고 동일 차원의 벡터 배열이 중심 → **인덱싱이 가장 빠르고 메모리 효율적**. LangChain wrapper가 Document를 pickle로 부가 저장해 메타/본문을 같이 복원.
- **필터링**: **post-filter** 방식(내부에서 `k * fetch_k_multiplier` 만큼 먼저 뽑은 뒤 메타로 걸러냄). 필터가 너무 엄격하면 결과가 k개보다 적게 돌아올 수 있다.
- **영속화**: `save_local(dir)` → `index.faiss`(벡터) + `index.pkl`(메타). `load_local` 시 `allow_dangerous_deserialization=True`가 필요할 정도로 **pickle 기반이라 신뢰된 파일만 로드**해야 함.

In [ ]:
from langchain_community.vectorstores import FAISS

FAISS_DIR = Path('./_store/faiss')
if FAISS_DIR.exists():
    shutil.rmtree(FAISS_DIR)
FAISS_DIR.parent.mkdir(parents=True, exist_ok=True)

t0 = time.time()
# 기본 인덱스는 Flat(IndexFlatL2) — 정확도 100%의 brute force. 규모가 커지면 다른 ANN 인덱스(IVF 등)로 교체 가능.
faiss_vs = FAISS.from_documents(chunks, emb)
faiss_vs.save_local(str(FAISS_DIR))   # index.faiss + index.pkl 두 파일이 생성됨
faiss_index_sec = time.time() - t0
print(f'[FAISS] index 시간: {faiss_index_sec:.3f}s, 문서 수: {faiss_vs.index.ntotal}')

t0 = time.time()
res = faiss_vs.similarity_search_with_score(q, k=3)
faiss_q_sec = time.time() - t0
print(f'[FAISS] query: {faiss_q_sec*1000:.1f}ms')
for doc, score in res:
    # FAISS도 기본은 L2 거리 (작을수록 유사)
    print(f'  L2={score:.3f} topic={doc.metadata["topic"]}  {doc.page_content[:60]}…')

In [ ]:
# FAISS 필터링 — 메타 topic=='finance' 인 청크만
# 주의: FAISS는 post-filter라 내부적으로 더 많은 후보를 뽑은 뒤 메타로 거른다.
# 필터가 너무 엄격하면 최종 결과가 k개 미만이 될 수 있다.
res = faiss_vs.similarity_search(q, k=3, filter={'topic': 'finance'})
print(f'finance 청크만: {len(res)}개 (요청 k=3)')
for d in res:
    print(f'  topic={d.metadata["topic"]} chunk_id={d.metadata["chunk_id"]}  {d.page_content[:60]}…')

## 3. Qdrant 🔒 (인메모리)

Rust로 작성된 **서버형 벡터 DB**. 프로토타이핑은 인메모리로, 운영은 Docker 컨테이너/바이너리로 동일 API를 쓸 수 있다는 것이 특징.

- **운영 모드 3종**
  - `QdrantClient(location=':memory:')` — 프로세스 내 메모리, 재시작 시 사라짐 (실습용).
  - `QdrantClient(path='./qdrant_local')` — 파일 기반 임베디드 모드 (서버 없이 영속).
  - `QdrantClient(url='http://qdrant.internal:6333')` — 서버 모드 (운영).
- **Payload 개념**: 각 포인트(벡터)에 JSON을 매달 수 있고, `Filter` DSL(must/should/must_not)로 복합 조건 검색 가능.
- **LangChain 주의**: `QdrantVectorStore`는 Document의 `metadata`를 payload의 `metadata` 키 아래에 넣는다. 그래서 필터 키가 `metadata.topic`처럼 **중첩 경로**로 쓰인다.

> 💡 실습은 `:memory:` 로 진행한다. 같은 코드에서 `location=':memory:'` 를 `url='http://...'` 로만 바꾸면 운영으로 이식된다 — 이게 Qdrant의 큰 장점.

In [ ]:
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http import models as qmodels

client = QdrantClient(location=':memory:')

# 임베딩 차원은 모델마다 다르므로(예: text-embedding-3-small=1536, bge-m3=1024)
# 한 번 호출해서 실제 크기를 측정 → 컬렉션 스펙과 불일치하면 적재 시 오류
vec_dim = len(emb.embed_query('dim probe'))

if client.collection_exists('efinance_terms'):
    client.delete_collection('efinance_terms')

client.create_collection(
    collection_name='efinance_terms',
    # distance는 cosine / dot / euclid 중 선택. 정규화된 임베딩 + cosine이 실무 표준.
    vectors_config=qmodels.VectorParams(size=vec_dim, distance=qmodels.Distance.COSINE),
)

t0 = time.time()
qdrant_vs = QdrantVectorStore(client=client, collection_name='efinance_terms', embedding=emb)
qdrant_vs.add_documents(chunks)
qdrant_index_sec = time.time() - t0
print(f'[Qdrant] index 시간: {qdrant_index_sec:.3f}s')

In [ ]:
t0 = time.time()
# Qdrant의 similarity_search_with_score는 컬렉션의 distance 설정에 따라 값의 의미가 달라진다.
# cosine이면 유사도(클수록 유사), euclid이면 거리(작을수록 유사).
res = qdrant_vs.similarity_search_with_score(q, k=3)
qdrant_q_sec = time.time() - t0
print(f'[Qdrant] query: {qdrant_q_sec*1000:.1f}ms')
for doc, score in res:
    print(f'  cos={score:.3f} topic={doc.metadata["topic"]}  {doc.page_content[:60]}…')

In [ ]:
# Qdrant 필터 DSL — (topic=finance) AND (chunk_id ≥ 2)
# LangChain wrapper는 Document.metadata를 payload의 'metadata' 키 아래에 중첩 저장하므로
# 필터 key가 'metadata.topic' / 'metadata.chunk_id' 형태가 된다. 이 prefix를 빼먹으면 매치 0건.
flt = qmodels.Filter(must=[
    qmodels.FieldCondition(key='metadata.topic', match=qmodels.MatchValue(value='finance')),
    qmodels.FieldCondition(key='metadata.chunk_id', range=qmodels.Range(gte=2)),
])
# Qdrant는 pre-filter(인덱스 탐색 전 payload로 먼저 거름) → 필터가 엄격해도 k개를 채우기 쉽다.
res = qdrant_vs.similarity_search(q, k=3, filter=flt)
for d in res:
    print(f'  chunk_id={d.metadata["chunk_id"]} topic={d.metadata["topic"]}  {d.page_content[:60]}…')

## 4. 세 스토어 종합 비교

아래 표는 이번 실습의 실측값이다. 절대값보다는 **상대 비교 관점**으로 읽자 — 실제 숫자는 임베딩 모델·코퍼스 크기·하드웨어에 크게 좌우된다.

> ⚠️ `query_ms`의 **스코어 의미는 스토어별로 다르다** (Chroma/FAISS = L2 거리, Qdrant = cosine 유사도). 시간만 비교하고 점수는 비교하지 말 것.

In [ ]:
import pandas as pd

summary = pd.DataFrame([
    {'store': 'Chroma', 'index_sec': round(chroma_index_sec, 3), 'query_ms': round(chroma_q_sec*1000, 1),
     'score_meaning': 'L2 distance (↓=유사)', 'persist': 'persist_directory', 'filter': 'JSON where/and/or'},
    {'store': 'FAISS',  'index_sec': round(faiss_index_sec, 3),  'query_ms': round(faiss_q_sec*1000, 1),
     'score_meaning': 'L2 distance (↓=유사)', 'persist': 'save_local', 'filter': 'dict (post-filter)'},
    {'store': 'Qdrant', 'index_sec': round(qdrant_index_sec, 3), 'query_ms': round(qdrant_q_sec*1000, 1),
     'score_meaning': 'cosine sim (↑=유사)', 'persist': ':memory: / path= / url=', 'filter': 'rich Filter DSL (pre-filter)'},
])
summary

## 정리

### 한눈에 보는 의사결정 가이드

| 상황 | 추천 |
|---|---|
| **노트북·실습·단일 사용자 프로토타입** | Chroma — 설치/사용이 가장 단순. 오늘 본 교재도 전부 Chroma. |
| **읽기 중심 대규모 배치 (수십만~백만 벡터)** | FAISS — 인덱싱·메모리 효율 최고, 메타 필터링은 단순한 경우에만. |
| **멀티 프로세스/서비스에서 동시 쓰기·복합 필터 필요** | Qdrant — 서버 모드로 전환 가능, Filter DSL이 강력. |

### 실무 체크포인트
- **영속화 파일은 인덱스일 뿐 원본이 아님** — 재인덱싱이 필요해지면 원문/청킹 파이프라인부터 재실행해야 한다. 청킹 전략이나 임베딩 모델이 바뀌면 반드시 **인덱스 재구축**.
- **Score 숫자를 임계값으로 직접 쓰지 말 것** — 스토어·distance 종류마다 스케일이 다르다. 임계값이 필요하면 해당 스토어·모델 조합으로 **경험적으로** 측정해 결정.
- **폐쇄망 이식**: Chroma/FAISS는 디렉터리를 통째로 복사해 반입. Qdrant는 서버 모드라면 스냅샷(`POST /collections/{name}/snapshots`) 을 반입하는 것이 표준.
- **필터 동작 차이**: FAISS는 **post-filter**라 엄격한 조건에서 결과가 k 미만이 될 수 있고, Qdrant는 **pre-filter**라 그런 문제가 없다. 필터가 중요한 도메인(권한/기밀 등급 등)에서는 Qdrant가 유리.

### 다음 노트북(2-3)과의 연결
`similarity_search`는 단일 쿼리에 대한 원샷 검색이다. 실제 RAG에서는 **MMR로 다양성 확보 / Hybrid(BM25+dense) / Self-Query로 메타 자동 추출** 같은 리트리버 패턴이 필요하다 — 2-3에서 같은 Chroma 인덱스 위에 올려본다.